# Estruturas de Dados — Camada Python sobre o Quadrante

## Vetores, Listas Encadeadas, Listas Lineares Restritas (Pilha e Fila)

Este notebook é o projeto da disciplina de **Estruturas de Dados**. Ele implementa,
**do zero** (sem usar `list.append`/`list.pop`/`collections.deque` como caixa-preta),
as quatro estruturas vistas em aula — **Vetor, Lista Encadeada, Pilha e Fila** — e as
aplica sobre um domínio real: um sistema de **roteirização e divisão territorial de
fiscais de campo** entre condomínios.

O domínio é uma versão simplificada e didática do **Quadrante**, um projeto meu (em Go,
na versão completa) que resolve o mesmo problema com algoritmos mais avançados (TSP,
k-means geográfico, alocação gulosa). Aqui a lógica foi reescrita em Python usando
exclusivamente as estruturas de dados clássicas trabalhadas na disciplina, para mostrar
onde cada uma se encaixa naturalmente em um sistema real:

| Estrutura | Onde entra no domínio | Por quê |
|---|---|---|
| **Vetor** | Cadastro de fiscais | Poucas inserções/remoções, muita leitura e percorrimento |
| **Lista Encadeada** | Carteira de condomínios de cada fiscal | Alta rotatividade: condomínios entram e saem o tempo todo (altas, baixas, realocações) |
| **Pilha (LIFO)** | Histórico de realocações, para desfazer | A última movimentação é sempre a primeira a ser desfeita |
| **Fila (FIFO)** | Solicitações de ausência pendentes | Processadas na ordem de chegada |

Os dados usados aqui são **totalmente fictícios** — nomes e endereços de exemplo,
sem qualquer relação com dados reais de clientes.

## 1. Vetor

Um vetor "de verdade" tem **capacidade fixa**: quando enche, é preciso alocar um bloco
maior e copiar tudo. É exatamente isso que a `list` do Python já faz por baixo dos panos
— mas aqui construímos essa lógica manualmente, sobre um bloco de tamanho fixo, para
deixar visível o que normalmente fica escondido.

```text
Capacidade 4, tamanho 3:
          ┌─────┬─────┬─────┬─────┐
Vetor:    │  F1 │  F2 │  F3 │  •  │   • = posição livre
          └─────┴─────┴─────┴─────┘
             0     1     2     3

Ao inserir o 4º elemento, o vetor enche. Nova inserção → redimensiona (dobra) e copia:

          ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
Vetor:    │  F1 │  F2 │  F3 │  F4 │  •  │  •  │  •  │  •  │
          └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘
```

In [1]:
class Vetor:
    """
    Vetor de tamanho dinâmico implementado sobre um array de capacidade fixa
    (a lista Python é usada só como bloco de memória bruto — cada operação é
    escrita na mão, como um vetor real: sem list.append/list.pop).

    Cresce por realocação (dobra de capacidade) quando enche, igual à
    estratégia usada internamente por list / ArrayList / std::vector.
    """

    def __init__(self, capacidade_inicial=4):
        self._dados = [None] * capacidade_inicial
        self._capacidade = capacidade_inicial
        self._tamanho = 0

    def __len__(self):
        return self._tamanho

    def __getitem__(self, indice):
        self._validar_indice(indice)
        return self._dados[indice]

    def __setitem__(self, indice, valor):
        self._validar_indice(indice)
        self._dados[indice] = valor

    def __iter__(self):
        for i in range(self._tamanho):
            yield self._dados[i]

    def _validar_indice(self, indice):
        if not (0 <= indice < self._tamanho):
            raise IndexError(f"índice {indice} fora do intervalo [0, {self._tamanho})")

    def _redimensionar(self, nova_capacidade):
        novo_bloco = [None] * nova_capacidade
        for i in range(self._tamanho):
            novo_bloco[i] = self._dados[i]
        self._dados = novo_bloco
        self._capacidade = nova_capacidade

    def inserir(self, valor):
        """Insere no final. O(1) amortizado — O(n) só nos raros momentos de realocação."""
        if self._tamanho == self._capacidade:
            self._redimensionar(self._capacidade * 2)
        self._dados[self._tamanho] = valor
        self._tamanho += 1

    def remover_no_indice(self, indice):
        """Remove por posição. O(n): desloca tudo à direita do índice uma casa para trás."""
        self._validar_indice(indice)
        removido = self._dados[indice]
        for i in range(indice, self._tamanho - 1):
            self._dados[i] = self._dados[i + 1]
        self._dados[self._tamanho - 1] = None
        self._tamanho -= 1
        return removido

    def buscar(self, predicado):
        """Busca linear O(n): retorna o índice do primeiro elemento que satisfaz o predicado, ou -1."""
        for i in range(self._tamanho):
            if predicado(self._dados[i]):
                return i
        return -1

    def pop(self):
        """Remove e retorna o ÚLTIMO elemento (equivalente ao list.pop() da aula). O(1)."""
        if self._tamanho == 0:
            raise IndexError("pop em vetor vazio")
        self._tamanho -= 1
        valor = self._dados[self._tamanho]
        self._dados[self._tamanho] = None
        return valor

    def inverter(self):
        """Inverte o vetor EM VIGOR (equivalente ao list.reverse() da aula). O(n), sem vetor auxiliar."""
        i, j = 0, self._tamanho - 1
        while i < j:
            self._dados[i], self._dados[j] = self._dados[j], self._dados[i]
            i += 1
            j -= 1

    def para_lista(self):
        return [self._dados[i] for i in range(self._tamanho)]

    def __repr__(self):
        return f"Vetor({self.para_lista()}, capacidade={self._capacidade})"

In [2]:
# Demonstração explícita: inserir, pop, buscar, inverter
v = Vetor(capacidade_inicial=2)
for x in [10, 20, 30]:
    v.inserir(x)
    print("inserir:", v)

print("\nremover_no_indice(1):", v.remover_no_indice(1))
print("estado:", v)

print("\npop() -> remove o último:", v.pop())
print("estado:", v)

for x in [40, 50, 60]:
    v.inserir(x)
print("\nantes de inverter:", v)
v.inverter()
print("depois de inverter():", v)

print("\nbuscar(valor == 50) -> índice:", v.buscar(lambda x: x == 50))

inserir: Vetor([10], capacidade=2)
inserir: Vetor([10, 20], capacidade=2)
inserir: Vetor([10, 20, 30], capacidade=4)

remover_no_indice(1): 20
estado: Vetor([10, 30], capacidade=4)

pop() -> remove o último: 30
estado: Vetor([10], capacidade=4)

antes de inverter: Vetor([10, 40, 50, 60], capacidade=4)
depois de inverter(): Vetor([60, 50, 40, 10], capacidade=4)

buscar(valor == 50) -> índice: 1


## 2. Lista Encadeada

Diferente do vetor, a lista encadeada não precisa de um bloco contíguo: cada nó guarda
seu valor e uma referência para o próximo. Inserir e remover não exigem deslocar nada —
só religar ponteiros.

```text
┌───────┬──┐     ┌───────┬──┐     ┌───────┬──────┐
│ Cond.1│ ─┼────>│ Cond.2│ ─┼────>│ Cond.3│ None │
└───────┴──┘     └───────┴──┘     └───────┴──────┘
```

**Por que a carteira de condomínios usa lista encadeada, e não vetor?** Porque ela sofre
inserção e remoção o tempo todo (alta/baixa de condomínio, realocação por ausência) e
quase nunca é acessada "pelo condomínio de índice 5" — é sempre percorrida ou buscada
por identificador. É exatamente o cenário em que a lista encadeada compensa o vetor.

In [3]:
class NoCondominio:
    def __init__(self, condominio):
        self.condominio = condominio
        self.proximo = None


class CarteiraEncadeada:
    """Carteira de condomínios de um fiscal, como lista encadeada simples."""

    def __init__(self, fiscal_id):
        self.fiscal_id = fiscal_id
        self._inicio = None
        self._tamanho = 0

    def __len__(self):
        return self._tamanho

    def inserir_fim(self, condominio):
        novo = NoCondominio(condominio)
        if self._inicio is None:
            self._inicio = novo
        else:
            atual = self._inicio
            while atual.proximo is not None:
                atual = atual.proximo
            atual.proximo = novo
        self._tamanho += 1

    def remover_por_id(self, condominio_id):
        """Remove o nó cujo condomínio tem o id dado. O(n). Retorna o condomínio removido ou None."""
        anterior = None
        atual = self._inicio
        while atual is not None:
            if atual.condominio.id == condominio_id:
                if anterior is None:
                    self._inicio = atual.proximo
                else:
                    anterior.proximo = atual.proximo
                self._tamanho -= 1
                return atual.condominio
            anterior = atual
            atual = atual.proximo
        return None

    def buscar_por_id(self, condominio_id):
        atual = self._inicio
        while atual is not None:
            if atual.condominio.id == condominio_id:
                return atual.condominio
            atual = atual.proximo
        return None

    def inverter(self):
        """
        Inverte a lista EM VIGOR (equivalente ao list.reverse() da aula, mas sem vetor
        auxiliar: religa os ponteiros um a um). O(n), O(1) de memória extra.
        """
        anterior = None
        atual = self._inicio
        while atual is not None:
            proximo = atual.proximo
            atual.proximo = anterior
            anterior = atual
            atual = proximo
        self._inicio = anterior

    def __iter__(self):
        atual = self._inicio
        while atual is not None:
            yield atual.condominio
            atual = atual.proximo

    def para_lista(self):
        return list(self)

    def __repr__(self):
        nomes = [c.nome for c in self]
        return f"CarteiraEncadeada(fiscal={self.fiscal_id}, {nomes})"

In [4]:
# Demonstração explícita: inserir_fim, remover_por_id, buscar_por_id, inverter
# (usamos aqui um stand-in simples porque a classe Condominio "de verdade" só é
# definida na seção 5 — o comportamento da lista encadeada não depende disso)
class _CondominioDemo:
    def __init__(self, id, nome):
        self.id = id
        self.nome = nome
    def __repr__(self):
        return self.nome

c1 = _CondominioDemo(1, "Residencial Alfa")
c2 = _CondominioDemo(2, "Residencial Beta")
c3 = _CondominioDemo(3, "Residencial Gama")

carteira = CarteiraEncadeada(fiscal_id=99)
for c in [c1, c2, c3]:
    carteira.inserir_fim(c)
    print("inserir_fim:", carteira)

print("\nbuscar_por_id(2):", carteira.buscar_por_id(2))

print("\nremover_por_id(2):", carteira.remover_por_id(2))
print("estado:", carteira)

carteira.inserir_fim(c2)  # devolve pra ilustrar o inverter() com 3 elementos
print("\nantes de inverter:", carteira)
carteira.inverter()
print("depois de inverter():", carteira)

inserir_fim: CarteiraEncadeada(fiscal=99, ['Residencial Alfa'])
inserir_fim: CarteiraEncadeada(fiscal=99, ['Residencial Alfa', 'Residencial Beta'])
inserir_fim: CarteiraEncadeada(fiscal=99, ['Residencial Alfa', 'Residencial Beta', 'Residencial Gama'])

buscar_por_id(2): Residencial Beta

remover_por_id(2): Residencial Beta
estado: CarteiraEncadeada(fiscal=99, ['Residencial Alfa', 'Residencial Gama'])

antes de inverter: CarteiraEncadeada(fiscal=99, ['Residencial Alfa', 'Residencial Gama', 'Residencial Beta'])
depois de inverter(): CarteiraEncadeada(fiscal=99, ['Residencial Beta', 'Residencial Gama', 'Residencial Alfa'])


### Vetor × Lista Encadeada (no contexto do Quadrante)

| Característica | Vetor (cadastro de fiscais) | Lista Encadeada (carteira de condomínios) |
|---|---|---|
| Acesso por posição | Direto, O(1) | Precisa percorrer, O(n) |
| Inserção no fim | O(1) amortizado | O(n) (percorre até o fim) — poderia virar O(1) guardando ponteiro pro fim |
| Remoção | O(n) — desloca elementos | O(n) — mas sem deslocamento, só religa ponteiros |
| Frequência de mudança no domínio | Baixa (fiscal é cadastrado uma vez) | Alta (condomínio muda de titular com frequência) |
| Padrão de acesso no domínio | Percorrer todos (relatórios) | Buscar/remover por ID (realocação) |

## 3. Pilha (LIFO) — histórico de realocações

Uma pilha segue o princípio **LIFO — Last In, First Out**: o último elemento inserido é
o primeiro a sair. No Quadrante original (Go), toda realocação por ausência é
**reversível** quando a ausência expira. Aqui simplificamos isso para um **desfazer
manual**: cada realocação vai para o topo da pilha, e desfazer sempre reverte a
movimentação mais recente primeiro — o mesmo princípio do botão "Voltar" de um
navegador, só que aplicado a condomínios em vez de páginas.

```text
             ↓ EMPILHAR (realocação)

           ┌───────────────┐
Topo →     │ Cond.5→Diego  │  ← desfazer aqui primeiro
           ├───────────────┤
           │ Cond.3→Carla  │
           ├───────────────┤
           │ Cond.1→Diego  │
           └───────────────┘

             ↑ DESEMPILHAR (desfazer)
```

In [5]:
class NoPilha:
    def __init__(self, valor):
        self.valor = valor
        self.anterior = None


class Pilha:
    """
    Pilha LIFO implementada com lista encadeada própria (sem usar list.append
    como pilha por baixo dos panos).
    """

    def __init__(self):
        self._topo = None
        self._tamanho = 0

    def __len__(self):
        return self._tamanho

    def vazia(self):
        return self._tamanho == 0

    def empilhar(self, valor):
        novo = NoPilha(valor)
        novo.anterior = self._topo
        self._topo = novo
        self._tamanho += 1

    def desempilhar(self):
        if self.vazia():
            raise IndexError("pilha vazia")
        no = self._topo
        self._topo = no.anterior
        self._tamanho -= 1
        return no.valor

    def topo(self):
        if self.vazia():
            raise IndexError("pilha vazia")
        return self._topo.valor

    def __repr__(self):
        valores = []
        atual = self._topo
        while atual is not None:
            valores.append(atual.valor)
            atual = atual.anterior
        return f"Pilha(topo->base: {valores})"

In [6]:
# Demonstração explícita: empilhar, topo, desempilhar
pilha = Pilha()

for r in ["Cond.1 -> Diego", "Cond.3 -> Carla", "Cond.5 -> Diego"]:
    pilha.empilhar(r)
    print("empilhar:", pilha)

print("\ntopo() (não remove):", pilha.topo())

print("\ndesempilhar() ->", pilha.desempilhar())
print("estado:", pilha)

print("\ndesempilhar() ->", pilha.desempilhar())
print("estado:", pilha)

print("\nvazia()?", pilha.vazia())

empilhar: Pilha(topo->base: ['Cond.1 -> Diego'])
empilhar: Pilha(topo->base: ['Cond.3 -> Carla', 'Cond.1 -> Diego'])
empilhar: Pilha(topo->base: ['Cond.5 -> Diego', 'Cond.3 -> Carla', 'Cond.1 -> Diego'])

topo() (não remove): Cond.5 -> Diego

desempilhar() -> Cond.5 -> Diego
estado: Pilha(topo->base: ['Cond.3 -> Carla', 'Cond.1 -> Diego'])

desempilhar() -> Cond.3 -> Carla
estado: Pilha(topo->base: ['Cond.1 -> Diego'])

vazia()? False


## 4. Fila (FIFO) — solicitações de ausência pendentes

Uma fila segue o princípio **FIFO — First In, First Out**: quem chega primeiro é
atendido primeiro. Solicitações de ausência (férias, falta, licença) chegam ao
coordenador em ordem cronológica e devem ser processadas nessa mesma ordem — não faria
sentido redistribuir a carteira de quem avisou por último antes de quem avisou primeiro.

```text
entra →   D  C  B  A   → sai (A é processado primeiro)
```

A implementação usa ponteiros de início **e** fim, o que mantém `enfileirar` e
`desenfileirar` em O(1) — uma fila baseada em vetor puro pagaria O(n) para remover do
início (teria que deslocar tudo).

In [7]:
class NoFila:
    def __init__(self, valor):
        self.valor = valor
        self.proximo = None


class Fila:
    """Fila FIFO com ponteiros de início e fim — O(1) em enfileirar/desenfileirar."""

    def __init__(self):
        self._inicio = None
        self._fim = None
        self._tamanho = 0

    def __len__(self):
        return self._tamanho

    def vazia(self):
        return self._tamanho == 0

    def enfileirar(self, valor):
        novo = NoFila(valor)
        if self._fim is None:
            self._inicio = novo
        else:
            self._fim.proximo = novo
        self._fim = novo
        self._tamanho += 1

    def desenfileirar(self):
        if self.vazia():
            raise IndexError("fila vazia")
        no = self._inicio
        self._inicio = no.proximo
        if self._inicio is None:
            self._fim = None
        self._tamanho -= 1
        return no.valor

    def frente(self):
        if self.vazia():
            raise IndexError("fila vazia")
        return self._inicio.valor

    def __repr__(self):
        valores = []
        atual = self._inicio
        while atual is not None:
            valores.append(atual.valor)
            atual = atual.proximo
        return f"Fila(frente->fim: {valores})"

In [8]:
# Demonstração explícita: enfileirar, frente, desenfileirar
fila = Fila()

for s in ["Bruno pediu férias", "Carla avisou falta", "Diego pediu licença"]:
    fila.enfileirar(s)
    print("enfileirar:", fila)

print("\nfrente() (não remove):", fila.frente())

print("\ndesenfileirar() ->", fila.desenfileirar())
print("estado:", fila)

print("\ndesenfileirar() ->", fila.desenfileirar())
print("estado:", fila)

print("\nvazia()?", fila.vazia())

enfileirar: Fila(frente->fim: ['Bruno pediu férias'])
enfileirar: Fila(frente->fim: ['Bruno pediu férias', 'Carla avisou falta'])
enfileirar: Fila(frente->fim: ['Bruno pediu férias', 'Carla avisou falta', 'Diego pediu licença'])

frente() (não remove): Bruno pediu férias

desenfileirar() -> Bruno pediu férias
estado: Fila(frente->fim: ['Carla avisou falta', 'Diego pediu licença'])

desenfileirar() -> Carla avisou falta
estado: Fila(frente->fim: ['Diego pediu licença'])

vazia()? False


## 5. Domínio

Versões simplificadas das entidades do Quadrante original (`internal/domain` em Go):
`Fiscal`, `Condominio`, e dois eventos — `SolicitacaoAusencia` (o que entra na fila) e
`Realocacao` (o que entra na pilha de histórico).

In [9]:
class Fiscal:
    def __init__(self, id, nome, papel="fiscal_campo"):
        self.id = id
        self.nome = nome
        self.papel = papel  # "fiscal_campo" ou "coordenador"

    def __repr__(self):
        return f"Fiscal({self.id}, {self.nome}, {self.papel})"


class Condominio:
    def __init__(self, id, nome, endereco, fiscal_titular_id):
        self.id = id
        self.nome = nome
        self.endereco = endereco
        self.fiscal_titular_id = fiscal_titular_id

    def __repr__(self):
        return f"Condominio({self.id}, {self.nome} -> fiscal {self.fiscal_titular_id})"


class SolicitacaoAusencia:
    def __init__(self, fiscal_id, motivo):
        self.fiscal_id = fiscal_id
        self.motivo = motivo

    def __repr__(self):
        return f"SolicitacaoAusencia({self.fiscal_id}, {self.motivo})"


class Realocacao:
    def __init__(self, condominio_id, fiscal_origem_id, fiscal_destino_id):
        self.condominio_id = condominio_id
        self.fiscal_origem_id = fiscal_origem_id
        self.fiscal_destino_id = fiscal_destino_id

    def __repr__(self):
        return f"Realocacao({self.condominio_id}: {self.fiscal_origem_id} -> {self.fiscal_destino_id})"

## 6. Sistema Quadrante — juntando as quatro estruturas

`SistemaQuadrante` é o "orquestrador": guarda os fiscais em um **Vetor**, a carteira de
cada fiscal em uma **Lista Encadeada**, as solicitações de ausência em uma **Fila** e o
histórico de realocações em uma **Pilha**.

A regra de negócio é uma alocação gulosa simplificada: ao processar uma ausência, cada
condomínio do fiscal ausente vai para quem estiver **com menos carga** no momento
(critério de desempate: menor ID, para ser determinístico) — a mesma ideia da alocação
gulosa do Quadrante original, sem o critério geográfico (que no projeto em Go vem de
Haversine sobre coordenadas reais).

In [10]:
class SistemaQuadrante:
    def __init__(self):
        self.fiscais = Vetor()
        self.carteiras = {}  # fiscal_id -> CarteiraEncadeada
        self.historico_realocacoes = Pilha()
        self.solicitacoes_pendentes = Fila()
        self._prox_condominio_id = 1

    def cadastrar_fiscal(self, id, nome, papel="fiscal_campo"):
        f = Fiscal(id, nome, papel)
        self.fiscais.inserir(f)
        self.carteiras[id] = CarteiraEncadeada(id)
        return f

    def cadastrar_condominio(self, nome, endereco, fiscal_titular_id):
        if fiscal_titular_id not in self.carteiras:
            raise ValueError(f"fiscal {fiscal_titular_id} não cadastrado")
        c = Condominio(self._prox_condominio_id, nome, endereco, fiscal_titular_id)
        self._prox_condominio_id += 1
        self.carteiras[fiscal_titular_id].inserir_fim(c)
        return c

    def buscar_fiscal(self, fiscal_id):
        indice = self.fiscais.buscar(lambda f: f.id == fiscal_id)
        return self.fiscais[indice] if indice != -1 else None

    def solicitar_ausencia(self, fiscal_id, motivo):
        self.solicitacoes_pendentes.enfileirar(SolicitacaoAusencia(fiscal_id, motivo))

    def _fiscal_menos_carregado(self, excluir_id):
        """Escolha gulosa e determinística: quem tem menos condomínios recebe a carteira."""
        melhor = None
        for f in self.fiscais:
            if f.id == excluir_id or f.papel == "coordenador":
                continue
            carga = len(self.carteiras[f.id])
            if melhor is None or carga < melhor[1] or (carga == melhor[1] and f.id < melhor[0].id):
                melhor = (f, carga)
        return melhor[0] if melhor else None

    def processar_proxima_ausencia(self):
        """
        Retira a solicitação mais antiga da fila (FIFO) e redistribui a
        carteira do fiscal ausente para quem está com menos carga. Cada
        movimentação vai para a pilha de histórico (LIFO), para poder ser
        desfeita na ordem inversa.
        """
        if self.solicitacoes_pendentes.vazia():
            return []

        solicitacao = self.solicitacoes_pendentes.desenfileirar()
        carteira_ausente = self.carteiras[solicitacao.fiscal_id]
        condominios = carteira_ausente.para_lista()

        realocacoes = []
        for c in condominios:
            destino = self._fiscal_menos_carregado(solicitacao.fiscal_id)
            if destino is None:
                break
            carteira_ausente.remover_por_id(c.id)
            c.fiscal_titular_id = destino.id
            self.carteiras[destino.id].inserir_fim(c)

            r = Realocacao(c.id, solicitacao.fiscal_id, destino.id)
            self.historico_realocacoes.empilhar(r)
            realocacoes.append(r)

        return realocacoes

    def desfazer_ultima_realocacao(self):
        """LIFO: desfaz sempre a movimentação mais recente primeiro."""
        if self.historico_realocacoes.vazia():
            return None

        r = self.historico_realocacoes.desempilhar()
        carteira_destino = self.carteiras[r.fiscal_destino_id]
        c = carteira_destino.remover_por_id(r.condominio_id)
        c.fiscal_titular_id = r.fiscal_origem_id
        self.carteiras[r.fiscal_origem_id].inserir_fim(c)
        return r

    def relatorio_carga(self):
        return {f.nome: len(self.carteiras[f.id]) for f in self.fiscais}

## 7. Demonstração completa

Cadastra 4 fiscais (1 coordenador + 3 de campo) e 7 condomínios fictícios, solicita a
ausência de um fiscal, processa a fila, mostra a pilha de histórico e desfaz a última
realocação.

In [11]:
sistema = SistemaQuadrante()

sistema.cadastrar_fiscal(1, "Ana", "coordenador")
sistema.cadastrar_fiscal(2, "Bruno")
sistema.cadastrar_fiscal(3, "Carla")
sistema.cadastrar_fiscal(4, "Diego")

for i in range(1, 6):
    sistema.cadastrar_condominio(f"Residencial {i}", f"Rua Fictícia, {i * 10}", fiscal_titular_id=2)
for i in range(1, 3):
    sistema.cadastrar_condominio(f"Parque {i}", f"Avenida Exemplo, {i * 20}", fiscal_titular_id=3)

print("Carga inicial:", sistema.relatorio_carga())

Carga inicial: {'Ana': 0, 'Bruno': 5, 'Carla': 2, 'Diego': 0}


In [12]:
sistema.solicitar_ausencia(2, "ferias")
print("Fila de solicitações antes de processar:", sistema.solicitacoes_pendentes)

realocacoes = sistema.processar_proxima_ausencia()
print("\nRealocações feitas (ordem cronológica):")
for r in realocacoes:
    print(" ", r)

print("\nCarga após ausência de Bruno:", sistema.relatorio_carga())
print("Pilha de histórico (topo = mais recente):", sistema.historico_realocacoes)

Fila de solicitações antes de processar: Fila(frente->fim: [SolicitacaoAusencia(2, ferias)])

Realocações feitas (ordem cronológica):
  Realocacao(1: 2 -> 4)
  Realocacao(2: 2 -> 4)
  Realocacao(3: 2 -> 3)
  Realocacao(4: 2 -> 4)
  Realocacao(5: 2 -> 3)

Carga após ausência de Bruno: {'Ana': 0, 'Bruno': 0, 'Carla': 4, 'Diego': 3}
Pilha de histórico (topo = mais recente): Pilha(topo->base: [Realocacao(5: 2 -> 3), Realocacao(4: 2 -> 4), Realocacao(3: 2 -> 3), Realocacao(2: 2 -> 4), Realocacao(1: 2 -> 4)])


In [13]:
desfeita = sistema.desfazer_ultima_realocacao()
print("Desfazendo a última realocação:", desfeita)
print("Carga após desfazer 1:", sistema.relatorio_carga())
print("Pilha após desfazer:", sistema.historico_realocacoes)

Desfazendo a última realocação: Realocacao(5: 2 -> 3)
Carga após desfazer 1: {'Ana': 0, 'Bruno': 1, 'Carla': 3, 'Diego': 3}
Pilha após desfazer: Pilha(topo->base: [Realocacao(4: 2 -> 4), Realocacao(3: 2 -> 3), Realocacao(2: 2 -> 4), Realocacao(1: 2 -> 4)])


## 8. Exercícios

1. **Vetor:** adicione um método `Vetor.reduzir_se_necessario()` que encolhe a
   capacidade pela metade quando o tamanho cai abaixo de 1/4 da capacidade (evita
   desperdiçar memória depois de muitas remoções).
2. **Lista Encadeada:** implemente `CarteiraEncadeada.inserir_inicio(condominio)` e
   explique em um comentário por que, ao contrário do vetor, essa operação é O(1).
3. **Pilha:** implemente um método `Pilha.desfazer_ate(n)` que desempilha e desfaz as
   últimas `n` realocações de uma vez, reaproveitando `desfazer_ultima_realocacao`.
4. **Fila:** adicione prioridade simples: solicitações com `motivo == "falta"` (ausência
   não planejada) devem furar a fila e ser processadas antes das de `"ferias"`. Dica:
   duas filas (urgente e normal) já resolvem, sem precisar de fila de prioridade real.
5. **Desafio:** una vetor + lista encadeada + hash: mantenha, junto do `Vetor` de
   fiscais, um dicionário `id -> índice no vetor` para tornar `buscar_fiscal` O(1) em vez
   de O(n) — e explique a troca de espaço por tempo que isso representa.

In [14]:
# Espaço para as soluções dos exercícios acima.
